# Phân tích phương án gộp nhiều mã causal_data

426 khóa product–store–week khớp nhiều dòng trong transaction DAY ≤ 537. Notebook so sánh các cách **mô tả** cờ nhị phân, không đưa feature vào mô hình và không xác nhận household đã tiếp xúc quảng cáo. Chỉ khảo sát các khóa trùng transaction train; không suy rộng cho mọi khóa causal_data.

## 1. Mã nguồn và phạm vi

Theo User Guide trang 6, `display=0` là không trưng bày và `mailer=0` là không có trong quảng cáo. Các mã display khác là vị trí trưng bày, không có thứ bậc đã xác nhận; không dùng `max` làm độ mạnh. Một khóa không khớp nguồn causal là trạng thái **chưa rõ**, không mặc định thành 0.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display
ROOT=Path.cwd()
if ROOT.name == 'research' and ROOT.parent.name == 'notebooks': ROOT = ROOT.parent
if ROOT.name=='notebooks':ROOT=ROOT.parent
DATA=ROOT/'artifacts'/'research'/'causal_context_audit';OUT=ROOT/'artifacts'/'research'/'causal_rule_sensitivity'
keys=['PRODUCT_ID','STORE_ID','WEEK_NO']
rows=pd.read_csv(DATA/'duplicate_causal_rows.csv',dtype={'display':'string','mailer':'string'},keep_default_na=False)
assert len(rows)==852 and rows[keys].drop_duplicates().shape[0]==426
assert not rows[['display','mailer']].eq('').any().any()
tx=pd.read_csv(ROOT/'sample_data'/'dunnhumby'/'transaction_data.csv',usecols=keys+['DAY'])
tx=tx.loc[tx.DAY<=537,keys]
assert not tx[keys].isna().any().any()
print('Khóa nhiều dòng:',rows[keys].drop_duplicates().shape[0],'| Dòng causal:',len(rows))

Khóa nhiều dòng: 426 | Dòng causal: 852


## 2. Các quy tắc mô tả

- **ANY:** ít nhất một dòng có mã khác `0`.
- **ALL:** mọi dòng đều có mã khác `0`.
- **Bất đồng:** ít nhất hai mã khác nhau; giữ trạng thái này riêng, không tự chọn một mã.

ANY và ALL chỉ là hai cách tóm tắt các dòng đã ghi nhận, không phải cận xác suất hay bằng chứng về sự thật. Nếu cùng có mã `0` và mã khác `0`, hai quy tắc cho kết quả khác nhau. Nếu hai vị trí trưng bày khác nhau nhưng đều khác `0`, cờ nhị phân giống nhau nhưng thông tin vị trí vẫn bất đồng.

In [2]:
groups=[]
for key,g in rows.groupby(keys,sort=False):
    d=set(g.display);m=set(g.mailer)
    groups.append(dict(zip(keys,key),display_codes='|'.join(sorted(d)),mailer_codes='|'.join(sorted(m)),
                       display_distinct=len(d),mailer_distinct=len(m),
                       display_any=any(x!='0' for x in d),display_all=all(x!='0' for x in d),
                       mailer_any=any(x!='0' for x in m),mailer_all=all(x!='0' for x in m)))
rules=pd.DataFrame(groups)
assert len(rules)==426 and not rules.duplicated(keys).any()
assert (rules.display_all<=rules.display_any).all() and (rules.mailer_all<=rules.mailer_any).all()
summary=pd.Series({'khóa nhiều mã':len(rules),
                   'display ANY khác ALL':int(rules.display_any.ne(rules.display_all).sum()),
                   'mailer ANY khác ALL':int(rules.mailer_any.ne(rules.mailer_all).sum()),
                   'display khác mã nhưng ANY = ALL':int(((rules.display_distinct>1)&(rules.display_any==rules.display_all)).sum())})
display(summary.to_frame('số khóa'))
display(rules.groupby(['display_codes','mailer_codes'],dropna=False).size().sort_values(ascending=False).head(15).rename('số khóa').to_frame())

,số khóa
khóa nhiều mã,426
display ANY khác ALL,426
mailer ANY khác ALL,1
display khác mã nhưng ANY = ALL,0


số khóa
display_codes mailer_codes         
0|1           A                 116
0|7           D                  79
              A                  74
0|6           D                  27
0|9           A                  15
0|3           A                  15
0|2           A                  13
0|5           A                  13
0|1           D                  12
0|9           D                   8
0|7           H                   7
0|6           A                   7
0|A           A                   5
0|4           A                   5
0|1           H                   4

## 3. Ảnh hưởng lên các dòng transaction

Nối bằng bảng khóa đã tổng hợp một dòng mỗi khóa, kiểm tra `many_to_one`. Các số dưới đây là dòng transaction DAY ≤ 537 có khóa nhiều mã, không phải household hay độ bao phủ chiến dịch. Chưa thay đổi dữ liệu nguồn.

In [3]:
joined=tx.merge(rules,on=keys,how='inner',validate='many_to_one')
assert len(joined)==587
summary_tx=pd.Series({'dòng transaction có khóa nhiều mã':len(joined),
                      'dòng có display ANY khác ALL':int(joined.display_any.ne(joined.display_all).sum()),
                      'dòng có mailer ANY khác ALL':int(joined.mailer_any.ne(joined.mailer_all).sum())})
display(summary_tx.to_frame('số dòng'))

,số dòng
dòng transaction có khóa nhiều mã,587
dòng có display ANY khác ALL,587
dòng có mailer ANY khác ALL,1


**Kết quả:** ANY và ALL cho kết quả display khác nhau ở cả 426 khóa, tương ứng 587 dòng transaction train. Với mailer, hai quy tắc khác nhau ở một khóa và một dòng transaction. Kết quả này nhấn mạnh rằng quy tắc gộp làm thay đổi giá trị; chưa cho biết quy tắc nào mô tả đúng thực tế.


## 4. Quyết định nghiên cứu

Ưu tiên **giữ riêng trạng thái bất đồng và không khớp** khi khảo sát; chưa dùng tính năng làm input. Có thể báo ANY/ALL như phân tích độ nhạy, nhưng không gọi ANY là “conservative” hay tự giữ mã đầu tiên. Không có căn cứ gán thứ bậc cho mã vị trí.

Khoảng ngày/tuần được quan sát không thể xác nhận thời điểm báo cáo được phát hành hoặc độ trễ. Giả định lag một tuần là kịch bản cần thử, không phải kết luận; còn cần nguồn xác nhận availability tại cutoff. Không suy ra tác động của quảng cáo hoặc tiếp xúc thực tế.

In [4]:
OUT.mkdir(parents=True,exist_ok=True)
rules.to_csv(OUT/'multi_code_rule_comparison.csv',index=False,encoding='utf-8-sig')
summary.to_csv(OUT/'key_summary.csv',header=True,encoding='utf-8-sig')
summary_tx.to_csv(OUT/'transaction_summary.csv',header=True,encoding='utf-8-sig')
print('Đã lưu:',OUT)

Đã lưu: artifacts\causal_rule_sensitivity
